In [187]:
import duckdb
import pandas as pd
from pathlib import Path
import gc
import yaml
import logging
from collections import defaultdict, deque
from typing import Any, List, Tuple, Optional
import re
import math

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', True)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True 
)


In [188]:
class ConnectionManager():
    def __init__(self, db_con_str):
        # la fonction duckdb.connect transforme le path en absolue, autant le faire ici 
        p = Path(db_con_str).expanduser().resolve()
        p.parent.mkdir(parents=True, exist_ok=True) # s'assurer que toute l'arboressence est créée

        self.db_con_str = p
        
        # connexion lazy
        self._con = None


    @property
    def con(self):
        if(self._con is None):
            # se connecter à la base de données
            self._con = duckdb.connect(self.db_con_str)

        return self._con


    @con.setter
    def con(self, value):
        # si au moment de changer la connexion on a déjà une connexion active
        if(self._con is not None):
            self._con.close() # cloturer la connexion en cours
            self._con = None # retirer la référence sur la connexion en cours
            gc.collect() # appeler le garbage collector pour forcer l'action de libérer les ressources et éviter les conflits d'accès

        self._con = value # pointer sur la nouvelle connexion
    
    
    def close_con(self):
        # on exploite le setter de la propriété pour cloturer correctement la connexion
        self.con = None


    def __del__(self):
        """Ferme automatiquement la connexion DuckDB quand l'objet est détruit."""
        try:
            self.close_con()
        except Exception:
            pass


    def __enter__(self):
        return self
    

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close_con()
        return False

In [189]:
class ConnectionUtils(ConnectionManager):
    def __init__(self, db_con_str : str):
        super().__init__(db_con_str)


    def tables(self):
        """Retourne la liste de toutes les tables physiques de la base courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            AND table_type = 'BASE TABLE'
            ORDER BY table_name
        """)


    def views(self):
        """Cette fonction renvoi la liste de toutes vues accessibles dans la base de données courante"""
        return self.con.sql("""
            SELECT table_name
            FROM information_schema.views
            WHERE table_catalog = current_database()
            ORDER BY table_name
        """)


    def tables_views(self):
        """Retourne la liste des tables et des vues de la base courante"""
        return self.con.sql("""
            SELECT 
                table_name,
                table_type          -- 'BASE TABLE' ou 'VIEW'
            FROM information_schema.tables
            WHERE table_catalog = current_database()
            ORDER BY table_type, table_name
        """)


    def table_exists(self, table_name : str):
        """Cette foction vérife qu'une table physique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{table_name}')
                AND
                (table_type = 'BASE TABLE')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]
    
    
    def view_exists(self, view_name : str):
        """Cette fonction check si une vue existe"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.views 
            WHERE 
                (table_name = '{view_name}')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def table_view_exists(self, name : str):
        """Cette foction vérife qu'une table physique ou une vue logique existe dans la base courante"""
        return self.con.sql(f"""
            SELECT COUNT(*) > 0 AS existe
            FROM information_schema.tables 
            WHERE 
                (table_name = '{name}')
                AND
                (table_type = 'BASE TABLE' OR table_type = 'VIEW')
                AND
                (table_catalog = current_database())
        """).fetchone()[0]


    def drop_table_if_exists(self, table_name : str):
        """Cette fonction permet de supprimer une table s'elle existe"""
        self.con.sql(f"DROP TABLE IF EXISTS {table_name}")


    def drop_tables_if_exists(self, tables : list[str]):
        """Cette fonction surpprime chaque table de la liste tables s'elle existe dans la base courante"""
        for table_name in tables : 
            self.drop_table_if_exists(table_name)


    def drop_view_if_exists(self, view_name : str):
        """Cette fonction permet de supprimer une vue s'elle existe"""
        self.con.sql(f"DROP VIEW IF EXISTS {view_name}")


    def drop_views_if_exists(self, views : list[str]):
        """Cette fonction surpprime chaque vue de la liste views s'elle existe dans la base courante"""
        for view_name in views : 
            self.drop_view_if_exists(view_name)


    def table(self, table_name : str):
        """Cette fonction renvoi la table dont le nom est passé en paramètre"""
        return self.con.table(table_name)


    def view(self, view_name : str):
        """Cette fonction renvoi la vue dont le nom est passé en paramètre"""
        return self.con.view(view_name)


    def table_view(self, name : str):
        """Retourne la relation d'une table ou d'une vue selon ce qui existe."""
        return self.con.sql(f"SELECT * FROM {name}")


    def create_table_view_if_not_exists(self, name: str, sql: str, type: str = "VIEW"):
        """
        Crée une vue ou une table uniquement si elle n'existe pas encore.
        """
        sql = sql.strip().rstrip(";")
        self.con.sql(f"""CREATE {type} IF NOT EXISTS {name} AS ({sql})""")
        

In [190]:
class DependencyTree:
    def __init__(self, data: dict[str, dict[str, Any]]):
        """
        data : dictionnaire de la forme
        {
            "v_sales": {"requires": ["t_sales"], ...},
            "t_sales": {"requires": ["df_sales"], ...},
            ...
        }
        """
        self.data = data
        self.graph = self._build_graph()          # node -> list of dependencies
        self.reverse_graph = self._build_reverse_graph()  # node -> list of dependents

    def _build_graph(self) -> dict[str, list[str]]:
        return {
            name: config.get("requires", [])
            for name, config in self.data.items()
        }

    def _build_reverse_graph(self) -> dict[str, list[str]]:
        reverse = defaultdict(list)
        for node, deps in self.graph.items():
            for dep in deps:
                reverse[dep].append(node)
        return dict(reverse)

    # -------------------------------------------------------------------------
    # Informations de base
    # -------------------------------------------------------------------------
    def nodes(self) -> list[str]:
        """Retourne tous les nœuds du graphe."""
        return list(self.graph.keys())

    def dependencies(self, name: str) -> list[str]:
        """Retourne les dépendances directes d'un nœud."""
        return self.graph.get(name, [])

    def dependents(self, name: str) -> list[str]:
        """Retourne les nœuds qui dépendent directement de celui-ci."""
        return self.reverse_graph.get(name, [])

    def roots(self) -> list[str]:
        """Nœuds qui ne sont requis par personne."""
        all_deps = {dep for deps in self.graph.values() for dep in deps}
        return [n for n in self.graph if n not in all_deps]

    def leaves(self) -> list[str]:
        """Nœuds qui n'ont aucune dépendance."""
        return [n for n, deps in self.graph.items() if not deps]

    # -------------------------------------------------------------------------
    # Dépendances récursives
    # -------------------------------------------------------------------------
    def all_dependencies(self, name: str) -> list[str]:
        """Retourne toutes les dépendances (directes + indirectes) dans l'ordre topologique."""
        result = []
        visited = set()

        def dfs(node: str):
            if node in visited:
                return
            visited.add(node)
            for dep in self.graph.get(node, []):
                dfs(dep)
            result.append(node)

        dfs(name)
        return result[:-1]  # on retire le nœud lui-même

    def creation_order(self, name: str | None = None) -> list[str]:
        """
        Ordre de création (topologique).
        Si name est fourni → uniquement pour ce nœud et ses dépendances.
        Sinon → ordre global.
        """
        if name:
            nodes = self.all_dependencies(name) + [name]
        else:
            nodes = self.nodes()

        in_degree = {n: 0 for n in nodes}
        for n in nodes:
            for dep in self.graph.get(n, []):
                if dep in in_degree:
                    in_degree[n] += 1

        queue = deque([n for n, deg in in_degree.items() if deg == 0])
        order = []

        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent in self.reverse_graph.get(node, []):
                if dependent in in_degree:
                    in_degree[dependent] -= 1
                    if in_degree[dependent] == 0:
                        queue.append(dependent)

        return order

    # -------------------------------------------------------------------------
    # Affichage
    # -------------------------------------------------------------------------
    def print_tree(self, root: str | None = None):
        """Affiche l'arbre de dépendances en texte."""
        def _print(node: str, prefix: str = "", is_last: bool = True, visited: set | None = None):
            if visited is None:
                visited = set()

            connector = "└── " if is_last else "├── "
            print(f"{prefix}{connector}{node}")

            if node in visited:
                print(f"{prefix}{'    ' if is_last else '│   '}└── [cycle détecté]")
                return

            visited = visited | {node}
            deps = self.graph.get(node, [])
            new_prefix = prefix + ("    " if is_last else "│   ")

            for i, dep in enumerate(deps):
                _print(dep, new_prefix, i == len(deps) - 1, visited)

        if root:
            print(f"\n=== Dépendances de '{root}' ===\n")
            _print(root)
        else:
            print("\n=== Graphe complet ===\n")
            roots = self.roots()
            for i, r in enumerate(roots):
                _print(r, is_last=(i == len(roots) - 1))

    def print_levels(self, name: str | None = None):
        """Affiche les nœuds par niveau topologique."""
        order = self.creation_order(name)
        print(f"\n=== Ordre de création {'de ' + name if name else 'global'} ===\n")
        for i, node in enumerate(order, 1):
            print(f"{i:2d}. {node}")

In [191]:
class ConnectionPipeline(ConnectionUtils):
    def __init__(self, db_con_str : str, pipeline_file_path : str):
        super().__init__(db_con_str)

        p = Path(pipeline_file_path).expanduser().resolve()
        p.parent.mkdir(parents=True, exist_ok=True) # s'assurer que toute l'arboressence est créée
        
        self.pipeline_file_path = p
        self._pipeline = None
        self._tree = None


    def load_pipeline(self) -> dict:
        """
        Charge la définition du pipeline.

        - Si pipeline_file_path est un fichier YAML, charge ce fichier.
        - Si pipeline_file_path est un dossier, charge récursivement tous les
        fichiers .yaml et .yml, puis les fusionne dans un seul dictionnaire.
        """
        path = self.pipeline_file_path

        if not path.exists():
            raise FileNotFoundError(
                f"Le chemin du pipeline n'existe pas : {path}"
            )

        # Cas 1 : un seul fichier YAML
        if path.is_file():
            if path.suffix.lower() not in {".yaml", ".yml"}:
                raise ValueError(
                    f"Le fichier pipeline doit être un fichier YAML : {path}"
                )

            with path.open("r", encoding="utf-8") as file:
                pipeline = yaml.safe_load(file) or {}

            if not isinstance(pipeline, dict):
                raise ValueError(
                    f"Le contenu YAML doit être un dictionnaire : {path}"
                )

            return pipeline

        # Cas 2 : dossier contenant plusieurs fichiers YAML
        yaml_files = sorted(
            [
                *path.rglob("*.yaml"),
                *path.rglob("*.yml"),
            ],
            key=lambda file_path: str(file_path.relative_to(path)),
        )

        if not yaml_files:
            raise FileNotFoundError(
                f"Aucun fichier .yaml ou .yml trouvé dans : {path}"
            )

        merged_pipeline: dict = {}
        object_sources: dict[str, Path] = {}

        for yaml_file in yaml_files:
            with yaml_file.open("r", encoding="utf-8") as file:
                current_pipeline = yaml.safe_load(file) or {}

            if not isinstance(current_pipeline, dict):
                raise ValueError(
                    f"Le contenu YAML doit être un dictionnaire : {yaml_file}"
                )

            for object_name, config in current_pipeline.items():
                if object_name in merged_pipeline:
                    previous_file = object_sources[object_name]

                    raise ValueError(
                        f"L'objet '{object_name}' est défini plusieurs fois : "
                        f"'{previous_file}' et '{yaml_file}'."
                    )

                merged_pipeline[object_name] = config
                object_sources[object_name] = yaml_file

        return merged_pipeline


    @property
    def pipeline(self):
        if(self._pipeline is None):
            self._pipeline = self.load_pipeline()
        return self._pipeline


    @property
    def tree(self):
        if(self._tree is None):
            self._tree = DependencyTree(self.pipeline)
        return self._tree
    

    def df_from_file(self, file: str | Path, **kwargs) -> pd.DataFrame:
        """Charge un fichier en DataFrame selon son extension + options"""
        path = Path(file).expanduser().resolve()
        suffix = path.suffix.lower()

        if suffix in {".xlsx", ".xls", ".xlsm"}:
            return pd.read_excel(path, **kwargs)

        elif suffix == ".csv":
            return pd.read_csv(path, **kwargs)

        elif suffix == ".tsv":
            return pd.read_csv(path, sep="\t", **kwargs)

        elif suffix == ".json":
            return pd.read_json(path, **kwargs)

        elif suffix == ".parquet":
            return pd.read_parquet(path, **kwargs)

        else:
            raise ValueError(f"Extension non supportée : {suffix}")


    def df_from_file_config(self, config : dict):
        # On prépare les kwargs en enlevant les clés réservées
        reserved = {"type", "requires", "file"}
        kwargs = {k: v for k, v in config.items() if k not in reserved}

        return self.df_from_file(config["file"], **kwargs)


    def process_dataframe_type(self, name : str):
        if(name in self.pipeline):
            if(not self.table_view_exists(name)):
                config = self.pipeline[name]
                df = self.df_from_file_config(config)
                self.con.register(name, df)


    def process_table_view_type(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]
            self.create_table_view_if_not_exists(name, config["sql"], config["type"])


    def process(self, name : str):
        logging.getLogger().debug(f"process({name})")

        if(name in self.pipeline):
            config = self.pipeline[name]

            if(config["type"] == "dataframe"):
                self.process_dataframe_type(name)

            elif(config["type"] in ["table", "view"]):
                self.process_table_view_type(name)


    def process_with_requires(self, name : str):
        if(name in self.pipeline):
            config = self.pipeline[name]

            if(not self.table_view_exists(name)):
                for subname in config.get("requires", []):
                    self.process_with_requires(subname)

                self.process(name)


    def p_table_view(self, name : str):
        self.process_with_requires(name)
        return self.table_view(name)

In [192]:

class SimUtils():
    @classmethod
    def optimal_removals_approx(cls,
        counts: List[int],
        target_proportions: List[float],
        max_prop_error: float = 0.02,
        min_removals: int = 0,          # ← nouveau paramètre
        tol: float = 1e-9
    ) -> Optional[Tuple[List[int], List[int], int, List[float]]]:
        """
        Version assouplie + contrainte de retraits minimum.

        Paramètres supplémentaires
        --------------------------
        min_removals : int
            Nombre minimum d'éléments à retirer au total.
            Utile pour éviter la solution triviale (0 retrait) 
            quand la répartition initiale est déjà proche de la cible.
        """
        n = len(counts)
        if len(target_proportions) != n:
            raise ValueError("counts et target_proportions doivent avoir la même longueur")
        if not math.isclose(sum(target_proportions), 1.0, abs_tol=tol):
            raise ValueError("La somme des proportions cibles doit être égale à 1")
        if min_removals < 0:
            raise ValueError("min_removals doit être ≥ 0")

        total_initial = sum(counts)

        # Borne supérieure du total restant
        candidates = [
            math.floor(c / p + tol)
            for c, p in zip(counts, target_proportions)
            if p > tol
        ]
        max_t = min(candidates) if candidates else 0

        # On force un nombre minimum de retraits
        max_t = min(max_t, total_initial - min_removals)

        if max_t <= 0:
            return None

        for t in range(max_t, 0, -1):
            ideal = [p * t for p in target_proportions]

            # Arrondi initial plafonné
            remaining = [min(counts[i], max(0, int(round(ideal[i])))) for i in range(n)]
            current_sum = sum(remaining)

            # Ajustement pour atteindre exactement t
            while current_sum > t:
                candidates = [
                    (remaining[i] - ideal[i], i)
                    for i in range(n) if remaining[i] > 0
                ]
                if not candidates:
                    break
                _, idx = max(candidates)
                remaining[idx] -= 1
                current_sum -= 1

            while current_sum < t:
                candidates = [
                    (ideal[i] - remaining[i], i)
                    for i in range(n) if remaining[i] < counts[i]
                ]
                if not candidates:
                    break
                _, idx = max(candidates)
                remaining[idx] += 1
                current_sum += 1

            if current_sum != t:
                continue

            # Vérification de l'erreur
            actual_props = [r / t for r in remaining]
            max_err = max(abs(actual_props[i] - target_proportions[i]) for i in range(n))

            if max_err <= max_prop_error:
                removals = [counts[i] - remaining[i] for i in range(n)]
                return removals, remaining, t, actual_props

        return None

In [193]:
cp = ConnectionPipeline("duckdb/pilotes/sim_v2/sim_v2.duckdb", "config")

In [194]:
# ============================================================================
# REFERENTIELS
# ============================================================================
 
display(cp.p_table_view("t_hotel_codes"))
display(cp.p_table_view("t_machines"))
display(cp.p_table_view("t_types"))
display(cp.p_table_view("t_gammes"))
display(cp.p_table_view("t_categories"))
display(cp.p_table_view("t_natures"))
display(cp.p_table_view("t_marques"))
display(cp.p_table_view("t_fournisseurs"))

# ============================================================================
# RANKING GLOBAL DES NATURES
# ============================================================================

display(cp.p_table_view("t_rank_nature"))

# ============================================================================
# RANKING DES NATURES PAR GROUPE
# ============================================================================

display(cp.p_table_view("t_rank_nature_by_type"))
display(cp.p_table_view("t_rank_nature_by_gamme"))
display(cp.p_table_view("t_rank_nature_by_categorie"))
display(cp.p_table_view("t_rank_nature_by_marque"))
display(cp.p_table_view("t_rank_nature_by_fournisseur"))

# ============================================================================
# RANKING DES GROUPES
# ============================================================================

display(cp.p_table_view("t_rank_type"))
display(cp.p_table_view("t_rank_gamme"))
display(cp.p_table_view("t_rank_categorie"))
display(cp.p_table_view("t_rank_marque"))
display(cp.p_table_view("t_rank_fournisseur"))


# =============================================================================
# DATASETS DE RÉFÉRENCE ET D'OBSERVATION
# =============================================================================

display(cp.p_table_view("t_dataset_ref"))

display(cp.p_table_view("t_dataset_observation_total"))

display(cp.p_table_view("t_dataset_observation_par_mois"))


# =============================================================================
# MIX D'EXPOSITION EN NOMBRES
# =============================================================================

display(cp.p_table_view("t_dataset_mix_nombre_global"))

display(cp.p_table_view("t_dataset_mix_nombre_long"))

display(cp.p_table_view("t_dataset_mix_nombre_detail"))

display(cp.p_table_view("t_dataset_mix_nombre"))


# =============================================================================
# MIX D'EXPOSITION EN POURCENTAGES
# =============================================================================

display(cp.p_table_view("t_dataset_mix_pourcentage_global"))

display(cp.p_table_view("t_dataset_mix_pourcentage_long"))

display(cp.p_table_view("t_dataset_mix_pourcentage_detail"))

display(cp.p_table_view("t_dataset_mix_pourcentage"))


# =============================================================================
# DATASET FINAL À UNE LIGNE PAR HÔTEL
# =============================================================================

display(cp.p_table_view("t_dataset_pivot"))

2026-08-05 09:55:02,319 | DEBUG | process(df_sales)


2026-08-05 09:56:03,332 | DEBUG | process(t_sales)
2026-08-05 09:56:04,349 | DEBUG | process(t_hotel_codes)


┌────────────┐
│ HOTEL_CODE │
│  varchar   │
├────────────┤
│ H0373      │
│ H2075      │
│ H3546      │
│ H5586      │
│ H6188      │
│ HB5I0      │
│ HB6A3      │
└────────────┘

2026-08-05 09:56:04,364 | DEBUG | process(t_machines)


┌─────────┐
│ MACHINE │
│ varchar │
├─────────┤
│ ARMOIRE │
│ BORNE   │
│ FRIGO   │
│ SCANNER │
└─────────┘

...

┌────────────┬─────────────┬────────────────────┬────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────┐
│ hotel_code │ fournisseur │   montant_marge    │ nombre_natures │                                                                                                                                                                                   natures                                                                                                                                                                                    │ rang_fournisseur │
│  varchar   │   varchar   │       double       │     int64      │                            

2026-08-05 09:56:04,972 | DEBUG | process(t_dataset_ref)


┌────────────┬───────────┬──────────────────┐
│ hotel_code │ solution  │ metres_lineaires │
│  varchar   │  varchar  │      double      │
├────────────┼───────────┼──────────────────┤
│ H0373      │ connected │              6.0 │
│ H2075      │ simply    │              6.0 │
│ H3546      │ connected │              7.0 │
│ H5586      │ connected │             NULL │
│ H6188      │ liberty   │              6.0 │
│ HB5I0      │ liberty   │              8.0 │
│ HB6A3      │ simply    │              2.0 │
└────────────┴───────────┴──────────────────┘

2026-08-05 09:56:04,992 | DEBUG | process(t_dataset_observation_total)


BinderException: Binder Error: Referenced column "NOMBRE_GUESTS" not found in FROM clause!
Candidate bindings: "HOTEL_GUESTS_PER_CHAMBRE", "NOM_BOUTIQUE", "NOM_PRODUIT", "COEF_MARGE", "NOM_PRODUIT_RAW"

LINE 10:     SUM(NOMBRE_GUESTS) AS nombre_guests,
                 ^

In [195]:
cp.table_view("df_sales")

┌──────────┬────────────┬──────────────────────────────────────────┬──────────────────┬───────────────────┬─────────────────┬──────────────────────────┬──────────────────────────────────────────┬──────────┬─────────┬─────────────┬─────────────┬────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────┬──────────────────────┬──────────────────┬─────────┬──────────────┬──────────────┬─────────────────┬─────────────┬─────────────┬─────────────┬─────────────────────┬──────────────┬─────────┬───────────────┬──────────┬───────────────────┬────────┬──────────┬──────────┬─────────────┬────────────┬───────────────────────────┬───────────────────┬─────────────────┬────────┐
│ SOLUTION │ HOTEL_CODE │                HOTEL_NAME                │ METRES_LINEAIRES │ HOTEL_NB_CHAMBRES │ HOTEL_TO_ANNUEL │ HOTEL_GUESTS_PER_CHAMBRE │               NOM_BOUTIQUE               │ TYPE_RAW │  TYPE   │  GAMME_RAW  │    GAMME    │                    